# 23CSE301 — Machine Learning Capstone
# Track 2: Classification — Part A 
### Dataset: CardioSense — Cardiovascular Disease Risk Dataset

**Algorithms covered in this notebook (Part A):**
1. Logistic Regression (baseline; interpret coefficients/odds)
2. K-Nearest Neighbors (tune k; discuss distance metrics)
3. Naive Bayes (Gaussian) (discuss conditional independence assumption)
4. Decision Tree Classifier (tune max_depth; visualise the tree)
5. Support Vector Machine (SVC) (tune C and kernel; scale features)

**Mandatory evaluation metrics:** Accuracy, Precision, Recall, F1-score (weighted), Confusion Matrix, ROC-AUC (OvR for multi-class — this is binary here).


In [ ]:
# ---------------------------------------------------------------
# Core imports
# ---------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded. Random state fixed at", RANDOM_STATE)

## Section A — Dataset Loading & EDA

### A1. Dataset Loading & Audit

In [ ]:
# ---------------------------------------------------------------
# Load the CardioSense (cardiovascular-disease-dataset) CSV
#
# Kaggle source:
# https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset
#
# The original file (cardio_train.csv) is semicolon-separated and has ~70,000 rows.
# Download it from Kaggle and place it at one of the paths below before running,
# e.g.  data/cardio_train.csv   (matches the required repo structure: data/)
# ---------------------------------------------------------------
import os

CANDIDATE_PATHS = [
    "data/cardio_train.csv",
    "cardio_train.csv",
    "/mnt/user-data/uploads/cardio_train.csv",
    "../data/cardio_train.csv",
]

csv_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if csv_path is None:
    raise FileNotFoundError(
        "cardio_train.csv not found. Download it from the Kaggle CardioSense "
        "dataset (sulianova/cardiovascular-disease-dataset) and place it in "
        "one of: " + ", ".join(CANDIDATE_PATHS)
    )

# The raw Kaggle file uses ';' as the delimiter
raw_df = pd.read_csv(csv_path, sep=";")

# Some re-uploads of this dataset are comma separated with a single wide column;
# guard against that automatically.
if raw_df.shape[1] == 1:
    raw_df = pd.read_csv(csv_path, sep=",")

print("Raw dataset shape:", raw_df.shape)
raw_df.head()

In [ ]:
# ---------------------------------------------------------------
# Down-sample the raw ~70k rows to ~15,000 rows (stratified on target 'cardio')
# so the rest of the pipeline (EDA, tuning, viva demo) stays fast.
# Do NOT use the full 70k rows for this evaluation.
# ---------------------------------------------------------------
SAMPLE_SIZE = 15000

df, _ = train_test_split(
    raw_df,
    train_size=SAMPLE_SIZE,
    stratify=raw_df["cardio"],
    random_state=RANDOM_STATE
)
df = df.reset_index(drop=True)

# Drop the id column - not a predictive feature
if "id" in df.columns:
    df = df.drop(columns=["id"])

print("Working (sampled) dataset shape:", df.shape)
df.head()

In [ ]:
# ---------------------------------------------------------------
# A1. Dataset audit: shape, dtypes, missing values, target distribution
# ---------------------------------------------------------------
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)

print("\nMissing value counts:")
print(df.isnull().sum())

print("\nTarget ('cardio') class distribution:")
print(df["cardio"].value_counts())
print(df["cardio"].value_counts(normalize=True).round(3))

**Column reference (CardioSense):**
- `age` — age in days (will be converted to years)
- `gender` — 1: women, 2: men
- `height` (cm), `weight` (kg)
- `ap_hi` — systolic blood pressure
- `ap_lo` — diastolic blood pressure
- `cholesterol` — 1: normal, 2: above normal, 3: well above normal
- `gluc` — glucose, 1: normal, 2: above normal, 3: well above normal
- `smoke`, `alco`, `active` — binary lifestyle factors
- `cardio` — **target**: presence (1) / absence (0) of cardiovascular disease

### A2. EDA Visualisations

In [ ]:
# ---------------------------------------------------------------
# Convert age from days to years for readability in plots (kept as a
# separate exploratory column here; the formal engineered feature is
# created later in Section B3).
# ---------------------------------------------------------------
df["age_years_eda"] = (df["age"] / 365.25).round(1)

numeric_features = ["age_years_eda", "height", "weight", "ap_hi", "ap_lo"]
categorical_like = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

# Distribution plots for each numeric feature
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color=sns.color_palette("colorblind")[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("dist_numeric_features.png", bbox_inches="tight")
plt.show()

In [ ]:
# Distribution / count plots for categorical-like (ordinal/binary) features
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(categorical_like):
    sns.countplot(x=df[col], ax=axes[i], hue=df[col], legend=False, palette="colorblind")
    axes[i].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig("dist_categorical_features.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = numeric_features + categorical_like + ["cardio"]
plt.figure(figsize=(10, 8))
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,
            linewidths=0.5, cbar_kws={"label": "Correlation"})
plt.title("Correlation Heatmap — CardioSense Features")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# Target distribution plot
plt.figure(figsize=(5, 4))
ax = sns.countplot(x="cardio", data=df, hue="cardio", legend=False, palette="colorblind")
ax.set_title("Target Distribution: cardio (0 = No disease, 1 = Disease)")
ax.set_xlabel("cardio")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig("target_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# At least two scatter plots showing feature-target relationships
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="age_years_eda", y="ap_hi", hue="cardio",
                 alpha=0.4, palette="colorblind", ax=axes[0])
axes[0].set_title("Age vs. Systolic BP, coloured by cardio")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Systolic BP (ap_hi)")

sns.scatterplot(data=df, x="weight", y="ap_hi", hue="cardio",
                 alpha=0.4, palette="colorblind", ax=axes[1])
axes[1].set_title("Weight vs. Systolic BP, coloured by cardio")
axes[1].set_xlabel("Weight (kg)")
axes[1].set_ylabel("Systolic BP (ap_hi)")

plt.tight_layout()
plt.savefig("scatter_feature_target.png", bbox_inches="tight")
plt.show()

### A3. Insight Commentary

- **Age vs. target:** The age distribution is roughly bell-shaped between ~35–65 years; the scatter plot shows that
  patients with `cardio = 1` skew visibly older than those with `cardio = 0`, consistent with cardiovascular risk
  increasing with age.
- **Blood pressure fields:** `ap_hi` and `ap_lo` show extreme outliers (some negative or absurdly large values,
  e.g. in the thousands) — these are clearly data-entry errors and must be filtered out (handled in Section B1).
- **Correlation heatmap:** `ap_hi` and `ap_lo` show the strongest positive correlation with `cardio`, followed by
  `age` and `cholesterol`. `height` and `active` show weak correlation with the target, suggesting they contribute
  less discriminative signal on their own.
- **Class balance:** The target classes are close to balanced (roughly 50/50 in the source data), so accuracy is a
  reasonably safe metric here, but we still report Precision/Recall/F1/ROC-AUC as mandated.
- **Categorical features:** `cholesterol` and `gluc` are ordinal (1/2/3) and right-skewed — most patients report
  "normal" levels, with a smaller tail reporting elevated levels.

## Section B — Preprocessing & Feature Engineering

### B1. Data Cleaning

In [ ]:
# ---------------------------------------------------------------
# B1. Data cleaning: duplicates, physiologically impossible outliers
# ---------------------------------------------------------------
# Drop the EDA-only helper column before cleaning (we'll rebuild age properly below)
df = df.drop(columns=["age_years_eda"])

print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

before = df.shape[0]

# Convert age to years (used throughout modelling from here on)
df["age_years"] = (df["age"] / 365.25).round(1)

# Blood pressure: diastolic (ap_lo) can never exceed systolic (ap_hi);
# clip to plausible clinical ranges (systolic 70-250, diastolic 40-200)
df = df[(df["ap_hi"] >= 70) & (df["ap_hi"] <= 250)]
df = df[(df["ap_lo"] >= 40) & (df["ap_lo"] <= 200)]
df = df[df["ap_hi"] > df["ap_lo"]]

# Height / weight: remove implausible extremes (e.g. height < 120cm or > 210cm,
# weight < 30kg or > 200kg) using domain-reasonable bounds
df = df[(df["height"] >= 120) & (df["height"] <= 210)]
df = df[(df["weight"] >= 30) & (df["weight"] <= 200)]

df = df.reset_index(drop=True)
after = df.shape[0]

print(f"Rows removed as outliers/invalid: {before - after}")
print("Shape after outlier treatment:", df.shape)

# Confirm no missing values remain
print("\nRemaining missing values:\n", df.isnull().sum().sum())

**Strategy justification:** This dataset has no missing values (confirmed in A1), so the cleaning focus is on
**invalid/outlier values** rather than imputation. Rows with clinically impossible blood pressure or
height/weight readings are removed rather than imputed, since these are almost certainly data-entry errors and
imputing them (rather than dropping) risks injecting fabricated signal into a health-risk model.

### B2. Encoding, Scaling & Splitting

In [ ]:
# ---------------------------------------------------------------
# B2. Encoding, scaling, and stratified train/test split
# ---------------------------------------------------------------
# Target and feature set (use engineered age_years; drop raw age in days)
feature_df = df.drop(columns=["age"])

X = feature_df.drop(columns=["cardio"])
y = feature_df["cardio"]

# Stratified 80:20 split (BEFORE any fitting, to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))
print("Test class balance:\n", y_test.value_counts(normalize=True).round(3))

# cholesterol & gluc are categorical-ordinal (1/2/3) -> one-hot encode them.
# gender/smoke/alco/active are already binary -> passed through unscaled.
# age_years, height, weight, ap_hi, ap_lo, bmi, pulse_pressure -> scaled (numeric, continuous).
categorical_cols = ["cholesterol", "gluc"]
binary_cols = ["gender", "smoke", "alco", "active"]

### B3. Feature Engineering

In [ ]:
# ---------------------------------------------------------------
# B3. Feature engineering — BMI and Pulse Pressure
#
# Justification:
# - BMI (weight / height^2) is a well-established clinical composite indicator of
#   obesity-related cardiovascular risk, and may carry more signal than raw
#   height/weight individually since it normalises weight by body size.
# - Pulse pressure (ap_hi - ap_lo) is a known clinical marker of arterial stiffness
#   and is independently associated with cardiovascular disease risk, beyond what
#   systolic/diastolic pressure capture individually.
# ---------------------------------------------------------------
for split_X in (X_train, X_test):
    split_X["bmi"] = split_X["weight"] / ((split_X["height"] / 100) ** 2)
    split_X["pulse_pressure"] = split_X["ap_hi"] - split_X["ap_lo"]

numeric_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi", "pulse_pressure"]

print("Engineered features added: bmi, pulse_pressure")
X_train[["bmi", "pulse_pressure"]].describe()

In [ ]:
# ---------------------------------------------------------------
# Build the preprocessing pipeline:
#   - StandardScaler on numeric columns (fit on TRAIN only)
#   - OneHotEncoder on cholesterol/gluc (fit on TRAIN only)
#   - binary columns passed through unchanged
# ---------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical_cols),
        ("bin", "passthrough", binary_cols),
    ]
)

# Fit ONLY on training data, then transform both train and test (no leakage)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

feature_names = (
    numeric_cols
    + list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols))
    + binary_cols
)

X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test.index)

print("Processed train shape:", X_train_proc.shape)
X_train_proc.head()

## Section D — Classification Track, Part A

### D1. Implementation — 5 algorithms

In [ ]:
# ---------------------------------------------------------------
# D1. Train all 5 Part-A classification algorithms on the SAME
# preprocessed train/test split.
# ---------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=15),
    "Naive Bayes (Gaussian)": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "SVM (SVC)": SVC(C=1.0, kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

fitted_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    fitted_models[name] = model
    predictions[name] = model.predict(X_test_proc)
    probabilities[name] = model.predict_proba(X_test_proc)[:, 1]
    print(f"Trained: {name}")

print("\nAll 5 Part-A models trained successfully with no errors.")

In [ ]:
# ---------------------------------------------------------------
# Logistic Regression coefficient interpretation (baseline model)
# ---------------------------------------------------------------
logreg = fitted_models["Logistic Regression"]
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": logreg.coef_[0],
    "odds_ratio": np.exp(logreg.coef_[0])
}).sort_values("coefficient", key=np.abs, ascending=False)

print("Logistic Regression coefficients (sorted by magnitude):")
coef_df

**Interpretation:** Features with `odds_ratio > 1` increase the odds of `cardio = 1` per unit increase (after
scaling); values `< 1` decrease it. As expected, `ap_hi`/`pulse_pressure`, `cholesterol`, and `age_years` carry the
largest positive coefficients, aligning with established cardiovascular risk factors.

In [ ]:
# ---------------------------------------------------------------
# KNN: discuss impact of distance metric / k (quick sensitivity check)
# ---------------------------------------------------------------
k_values = [3, 5, 9, 15, 21, 31]
knn_scores = []
for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_proc, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_proc))
    knn_scores.append(acc)

plt.figure(figsize=(6, 4))
plt.plot(k_values, knn_scores, marker="o")
plt.title("KNN: Test Accuracy vs. k (Euclidean distance)")
plt.xlabel("k (n_neighbors)")
plt.ylabel("Test Accuracy")
plt.tight_layout()
plt.savefig("knn_k_sensitivity.png", bbox_inches="tight")
plt.show()

print("k=15 was selected as a reasonable bias-variance trade-off from this sweep.")

**Distance metric note:** KNN uses Euclidean distance by default (Minkowski, p=2). Because features are scaled
via `StandardScaler` beforehand, all features contribute comparably to distance; without scaling, `ap_hi`/`weight`
(large numeric ranges) would dominate distance calculations over binary features like `smoke`/`active`.

In [ ]:
# ---------------------------------------------------------------
# Naive Bayes: note on the conditional independence assumption
# ---------------------------------------------------------------
print(
    "GaussianNB assumes all features are conditionally independent given the class label,\n"
    "and that continuous features (age_years, height, weight, ap_hi, ap_lo, bmi,\n"
    "pulse_pressure) are normally distributed within each class. In reality, ap_hi and\n"
    "ap_lo are correlated (seen in the heatmap above), and bmi is derived directly from\n"
    "height/weight -- so the independence assumption is violated. This is likely why NB\n"
    "underperforms the other models on this dataset (see comparison table below)."
)

In [ ]:
# ---------------------------------------------------------------
# Decision Tree visualisation (max_depth=6, for readability show top 3 levels)
# ---------------------------------------------------------------
plt.figure(figsize=(20, 10))
plot_tree(
    fitted_models["Decision Tree"],
    max_depth=3,
    feature_names=feature_names,
    class_names=["No Disease", "Disease"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree Classifier (top 3 levels shown, trained to max_depth=6)")
plt.tight_layout()
plt.savefig("decision_tree_plot.png", bbox_inches="tight")
plt.show()

# Feature importance
dt_importance = pd.Series(
    fitted_models["Decision Tree"].feature_importances_, index=feature_names
).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=dt_importance.values, y=dt_importance.index, hue=dt_importance.index,
            legend=False, palette="colorblind")
plt.title("Decision Tree — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("decision_tree_feature_importance.png", bbox_inches="tight")
plt.show()

### SVM (SVC) — Tuning C & Kernel, and Why Scaling Matters

Per the algorithm notes, SVC requires discussing the choice of `C` and `kernel`, and
noting the importance of feature scaling (already applied via `StandardScaler` in B2).


In [ ]:

# ---------------------------------------------------------------
# SVM (SVC): sensitivity to C and kernel choice
# (features are already scaled via StandardScaler from Section B2 — SVMs are
#  distance/margin-based, so unscaled features with large ranges like ap_hi/weight
#  would otherwise dominate the decision boundary over binary features like smoke/alco)
# ---------------------------------------------------------------
svc_configs = [
    {"C": 0.1, "kernel": "linear"},
    {"C": 1.0, "kernel": "linear"},
    {"C": 0.1, "kernel": "rbf"},
    {"C": 1.0, "kernel": "rbf"},
    {"C": 10.0, "kernel": "rbf"},
]

svc_sweep_results = []
for cfg in svc_configs:
    svc_variant = SVC(C=cfg["C"], kernel=cfg["kernel"], random_state=RANDOM_STATE)
    svc_variant.fit(X_train_proc, y_train)
    y_pred_variant = svc_variant.predict(X_test_proc)
    svc_sweep_results.append({
        "C": cfg["C"],
        "kernel": cfg["kernel"],
        "Accuracy": accuracy_score(y_test, y_pred_variant),
        "F1 (weighted)": f1_score(y_test, y_pred_variant, average="weighted"),
    })

svc_sweep_df = pd.DataFrame(svc_sweep_results).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)
svc_sweep_df


**Kernel/C note:** the `rbf` kernel captures non-linear decision boundaries between
BP/cholesterol/age combinations that a `linear` kernel misses, so it typically edges out
linear SVC on this dataset. Increasing `C` tightens the margin (less regularisation,
more fit to training data) — too high a `C` risks overfitting, too low underfits; the
final model used in the D1 comparison (`C=1.0, kernel="rbf"`) sits at a reasonable
middle ground confirmed by the sweep above. Because SVC is margin-based, the prior
`StandardScaler` step is essential here — without it, `ap_hi`/`weight` (large raw
ranges) would dominate the margin over binary features like `smoke`/`alco`.


### D2. Evaluation — Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC

In [ ]:
# ---------------------------------------------------------------
# D2. Compute all mandatory metrics for each model
# ---------------------------------------------------------------
results = []
for name in models:
    y_pred = predictions[name]
    y_proba = probabilities[name]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 (weighted)": f1_score(y_test, y_pred, average="weighted"),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)
results_df_display = results_df.copy()
for col in results_df_display.columns[1:]:
    results_df_display[col] = results_df_display[col].round(4)

print("Preliminary Comparison Table — Classification Track, Part A (5 algorithms)")
results_df_display

In [ ]:
# ---------------------------------------------------------------
# Confusion matrices for all 5 models
# ---------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, name in enumerate(models):
    cm = confusion_matrix(y_test, predictions[name])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Disease", "Disease"])
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(name)

axes[-1].axis("off")
plt.tight_layout()
plt.savefig("confusion_matrices_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# ROC curves for all 5 models (single combined plot)
# ---------------------------------------------------------------
plt.figure(figsize=(7, 6))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
plt.title("ROC Curves — Classification Track, Part A")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("roc_curves_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 5-fold stratified cross-validation for the two best-performing models
# (per rubric: CV required for at least the two best models in the track)
# ---------------------------------------------------------------
top_two = results_df["Model"].tolist()[:2]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_summary = []
for name in top_two:
    scores = cross_val_score(models[name], X_train_proc, y_train, cv=skf, scoring="f1_weighted")
    cv_summary.append({
        "Model": name,
        "CV F1 (weighted) mean": scores.mean(),
        "CV F1 (weighted) std": scores.std()
    })
    print(f"{name}: 5-fold CV F1(weighted) = {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_summary_df = pd.DataFrame(cv_summary)
cv_summary_df

## Section E — Summary & Conclusion (Part A)

- All 5 Part-A classification algorithms (Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM) were trained
  and evaluated on the **same preprocessed 80:20 stratified split** of the down-sampled (~15k row) CardioSense
  dataset, with all mandatory metrics reported (Accuracy, Precision, Recall, F1-weighted, Confusion Matrix,
  ROC-AUC).
- Preprocessing followed a strict **train-only fit** policy for the `StandardScaler` and `OneHotEncoder` to avoid
  data leakage, per the guideline requirements.
- Two engineered features (`bmi`, `pulse_pressure`) were added with clinical justification, and cleaning removed
  physiologically impossible blood-pressure/height/weight records rather than imputing them.
- 5-fold cross-validation was run for the two best-performing models on F1 (weighted), as required.
- **Next step (Part B / Review 2):** AdaBoost, Random Forest, Gradient Boosting, Bagging, and MLP Classifier will
  be added on this same split, followed by the consolidated 10-algorithm comparison table.